# Diabetes Prediction ML Project

Interactive notebook version for presentation and evaluation.

## 1. Imports & Setup

In [ ]:
"""
model_training.py — Full ML Pipeline for Diabetes Prediction
Trains 5 classifiers on the Pima Indians Diabetes Dataset,
selects the best (Random Forest), and saves model + scaler as .pkl files.
"""

import pandas as pd
import numpy as np
import joblib
import json
import os
from io import StringIO
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ─────────────────────────────────────────────────────────────────────────────

## 2. Original Training Pipeline
Run the cell below to execute your existing project code.

In [ ]:
"""
model_training.py — Full ML Pipeline for Diabetes Prediction
Trains 5 classifiers on the Pima Indians Diabetes Dataset,
selects the best (Random Forest), and saves model + scaler as .pkl files.
"""

import pandas as pd
import numpy as np
import joblib
import json
import os
from io import StringIO
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — DATA LOADING
# Pima Indians Diabetes Dataset (embedded as CSV string for portability)
# Original source: UCI ML Repository / Kaggle
# ─────────────────────────────────────────────────────────────────────────────

DIABETES_CSV = """Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
6,148,72,35,0,33.6,0.627,50,1
1,85,66,29,0,26.6,0.351,31,0
8,183,64,0,0,23.3,0.672,32,1
1,89,66,23,94,28.1,0.167,21,0
0,137,40,35,168,43.1,2.288,33,1
5,116,74,0,0,25.6,0.201,30,0
3,78,50,32,88,31.0,0.248,26,1
10,115,0,0,0,35.3,0.134,29,0
2,197,70,45,543,30.5,0.158,53,1
8,125,96,0,0,0.0,0.232,54,1
4,110,92,0,0,37.6,0.191,30,0
10,168,74,0,0,38.0,0.537,34,1
10,139,80,0,0,27.1,1.441,57,0
1,189,60,23,846,30.1,0.398,59,1
5,166,72,19,175,25.8,0.587,51,1
7,100,0,0,0,30.0,0.484,32,1
0,118,84,47,230,45.8,0.551,31,1
7,107,74,0,0,29.6,0.254,31,1
1,103,30,38,83,43.3,0.183,33,0
1,115,70,30,96,34.6,0.529,32,1
3,126,88,41,235,39.3,0.704,27,0
8,99,84,0,0,35.4,0.388,50,0
7,196,90,0,0,39.8,0.451,41,1
9,119,80,35,0,29.0,0.263,29,1
11,143,94,33,146,36.6,0.254,51,1
10,125,70,26,115,31.1,0.205,41,1
7,147,76,0,0,39.4,0.257,43,1
1,97,66,15,140,23.2,0.487,22,0
13,145,82,19,110,22.2,0.245,57,0
5,117,92,0,0,34.1,0.337,38,0
5,109,75,26,0,36.0,0.546,60,0
3,158,76,36,245,31.6,0.851,28,1
3,88,58,11,54,24.8,0.267,22,0
6,92,92,0,0,19.9,0.188,28,0
10,122,78,31,0,27.6,0.512,45,0
4,103,60,33,192,24.0,0.966,33,0
11,138,76,0,0,33.2,0.420,35,0
9,102,76,37,0,32.9,0.665,46,1
2,90,68,42,0,38.2,0.503,27,1
4,111,72,47,207,37.1,1.390,56,1
3,180,64,25,70,34.0,0.271,26,0
7,133,84,0,0,40.2,0.696,37,0
7,106,92,18,0,22.7,0.235,48,0
9,171,110,24,240,45.4,0.721,54,1
7,159,64,0,0,27.4,0.294,40,0
0,180,66,39,0,42.0,1.893,25,1
1,146,56,0,0,29.7,0.564,29,0
2,71,70,27,0,28.0,0.586,22,0
7,103,66,32,0,39.1,0.344,31,1
7,105,0,0,0,0.0,0.305,24,0
1,103,80,11,82,19.4,0.491,22,0
1,101,50,15,36,24.2,0.526,26,0
5,88,66,21,23,24.4,0.342,30,0
8,176,90,34,300,33.7,0.467,58,1
7,150,66,42,342,34.7,0.718,42,0
1,73,50,10,0,23.0,0.248,21,0
7,187,68,39,304,37.7,0.254,41,1
0,100,88,60,110,46.8,0.962,31,0
0,146,82,0,0,40.5,1.781,44,0
0,105,64,41,142,41.5,0.173,22,0
2,84,0,0,0,0.0,0.304,21,0
8,133,72,0,0,32.9,0.270,39,1
5,44,62,0,0,25.0,0.587,36,0
2,141,58,34,128,25.4,0.699,24,0
7,114,66,0,0,32.8,0.258,42,1
5,99,54,28,83,34.0,0.499,30,0
0,109,88,30,0,32.5,0.855,38,1
2,109,92,0,0,42.7,0.845,54,0
1,95,66,13,38,19.6,0.334,25,0
4,146,85,27,100,28.9,0.189,27,0
8,112,72,0,0,23.6,0.840,58,0
0,138,0,0,0,36.3,0.933,25,1
3,102,44,20,94,30.8,0.400,26,0
3,87,60,18,0,21.8,0.444,21,0
4,156,75,0,0,48.3,0.238,32,1
6,93,50,30,64,28.7,0.356,23,0
4,61,96,0,0,35.5,0.925,52,0
4,129,60,12,231,27.5,0.527,31,0
3,138,62,44,208,24.0,0.290,22,0
0,98,64,17,0,20.6,0.507,24,0
9,154,78,30,100,30.9,0.164,45,0
7,103,68,40,0,46.2,0.665,33,0
1,126,60,0,0,30.1,0.349,47,1
8,99,76,0,0,35.4,0.388,50,0
4,112,76,0,0,38.3,0.234,35,0
8,167,106,46,231,37.6,0.165,43,1
1,78,76,32,0,36.9,0.264,29,0
3,90,78,0,0,42.7,0.559,21,0
9,164,84,21,0,30.8,0.831,32,1
5,145,80,0,0,37.9,0.637,41,0
1,77,56,30,56,33.3,1.251,24,0
4,110,76,20,100,28.4,0.118,27,0
2,74,0,0,0,0.0,0.102,22,0
6,149,88,29,260,28.5,0.687,55,0
9,134,74,33,60,25.9,0.460,81,0
1,73,50,10,0,23.0,0.248,21,0
6,107,88,0,0,36.8,0.727,31,0
1,65,76,0,0,24.8,0.248,23,0
8,130,82,42,0,38.4,0.503,45,1
8,187,90,34,0,37.7,0.956,55,1
5,100,76,27,0,39.4,0.560,35,0
4,68,120,0,0,35.1,0.258,22,0
9,112,82,24,0,28.2,1.282,50,1
0,119,0,0,0,32.4,0.141,24,1
2,112,68,22,94,34.1,0.315,26,0
2,116,74,0,0,25.6,0.201,30,0
2,70,56,27,0,31.7,0.503,26,0
3,91,78,36,0,39.6,0.498,28,0
4,105,72,0,0,22.9,0.143,35,0
4,189,110,31,0,28.5,0.680,37,0
0,147,85,54,0,42.8,0.375,22,0
4,112,72,0,0,23.1,0.289,22,0
7,142,60,33,0,28.8,0.687,61,0
4,109,64,44,99,34.6,0.415,34,0
1,123,88,39,182,40.9,0.969,35,0
7,136,90,0,0,29.9,0.210,50,0
3,106,72,0,0,25.8,0.207,27,0
3,103,72,30,152,27.6,0.730,27,0
1,128,88,39,110,36.5,1.057,37,1
3,115,66,39,140,38.1,0.150,28,0
5,126,130,0,0,30.1,0.473,47,0
2,106,64,35,119,30.5,1.400,34,0
2,102,86,36,120,45.5,0.127,23,1
6,125,68,30,120,30.0,0.464,32,0
0,126,86,27,120,27.4,0.515,21,0
2,168,64,0,0,36.4,0.554,38,1
3,152,72,30,97,27.6,0.761,35,0
6,156,60,0,0,26.5,0.483,60,0
0,95,80,45,92,36.5,0.330,26,0
3,71,88,0,0,30.4,0.247,22,0
4,99,68,38,0,32.8,0.145,33,0
4,107,68,0,0,29.4,0.251,28,0
8,100,76,0,0,38.7,0.190,42,0
1,97,105,0,0,30.1,0.218,21,0
0,123,72,0,0,36.3,0.258,52,1
6,115,60,0,0,34.3,0.169,48,0
4,99,96,0,0,24.6,0.847,35,0
1,119,88,41,170,45.3,0.507,26,0
4,114,64,0,0,28.9,0.126,24,0
3,182,74,0,0,30.5,0.345,46,1
7,68,49,26,135,42.1,0.398,52,1
0,162,76,56,100,53.2,0.759,25,1
1,72,72,25,0,31.2,0.295,44,0
8,101,76,48,58,35.8,0.467,52,1
4,97,60,23,0,28.2,0.443,22,0
3,83,58,31,18,34.3,0.336,25,0
2,109,75,25,0,31.9,0.253,27,0
6,119,50,22,176,27.1,1.318,33,1
0,99,56,0,0,28.2,0.201,25,0
2,65,60,35,0,24.9,0.243,21,0
5,130,82,0,0,39.1,0.956,37,1
2,87,58,16,52,32.7,0.166,25,0
5,175,96,0,0,36.5,0.746,48,1
4,108,80,0,0,26.5,0.144,37,0
9,125,70,30,0,36.4,0.738,48,1
1,157,72,21,168,25.6,0.123,24,0
12,140,85,33,0,37.4,0.244,41,0
5,147,75,0,0,29.9,0.434,28,0
1,97,66,15,140,23.2,0.487,22,0
6,107,88,0,0,36.8,0.727,31,0
0,189,104,25,0,34.3,0.435,41,1
4,108,80,0,0,26.5,0.144,37,0
1,121,78,39,74,39.0,0.261,28,0
3,108,62,24,0,26.0,0.223,25,0
0,181,88,44,510,43.3,0.222,26,1
8,154,78,32,0,32.4,0.443,45,1
1,128,88,39,110,36.5,1.057,37,1
7,137,90,41,0,32.0,0.391,39,0
0,123,72,0,0,36.3,0.258,52,1
1,106,76,0,0,37.5,0.197,26,0
6,190,92,0,0,35.5,0.278,66,1
2,88,58,26,16,28.4,0.766,22,0
9,170,74,31,0,44.0,0.403,43,1
9,89,62,0,0,22.5,0.142,33,0
13,145,82,19,110,22.2,0.245,57,0
2,197,70,45,543,30.5,0.158,53,1
4,99,68,38,0,32.8,0.145,33,0
3,158,76,36,245,31.6,0.851,28,1
6,92,92,0,0,19.9,0.188,28,0
10,122,78,31,0,27.6,0.512,45,0
4,103,60,33,192,24.0,0.966,33,0
0,100,88,60,110,46.8,0.962,31,0
9,171,110,24,240,45.4,0.721,54,1
8,126,88,36,108,38.5,0.349,49,0
5,123,74,40,77,34.1,0.269,28,0
0,107,60,25,0,26.4,0.133,23,0
6,111,64,39,0,34.2,0.260,24,0
3,76,68,15,112,25.0,0.628,39,0
8,74,64,46,0,21.0,0.672,29,1
1,97,66,15,140,23.2,0.487,22,0
5,116,74,0,0,25.6,0.201,30,0
1,126,60,0,0,30.1,0.349,47,1
3,78,50,32,88,31.0,0.248,26,1
1,101,62,0,0,21.9,0.336,28,0
7,168,88,42,321,38.2,0.787,40,1
7,159,64,0,0,27.4,0.294,40,0
0,155,84,0,0,34.4,0.304,23,0
0,118,84,47,230,45.8,0.551,31,1
3,156,84,35,220,28.3,0.884,27,1
7,103,66,32,0,39.1,0.344,31,1
6,111,74,0,0,26.4,0.206,40,0
6,139,80,57,0,52.7,1.695,22,1
4,112,72,0,0,23.1,0.289,22,0
5,100,76,27,0,39.4,0.560,35,0
9,127,70,39,0,33.7,0.537,54,0
4,129,86,20,270,35.1,0.231,23,0
6,170,80,46,0,29.0,0.263,52,0
1,97,60,23,0,28.2,0.443,22,0
7,181,84,21,192,35.9,0.586,51,1
0,117,80,31,53,45.2,0.089,24,0
3,83,58,31,18,34.3,0.336,25,0
2,141,58,34,128,25.4,0.699,24,0
1,97,68,21,0,27.2,1.095,22,0
4,83,86,19,0,29.3,0.317,34,0
0,101,64,17,0,21.0,0.252,21,0
3,112,78,50,140,39.4,0.175,24,0
4,152,76,0,0,33.2,0.731,47,1
2,109,75,25,0,31.9,0.253,27,0
10,139,80,0,0,27.1,1.441,57,0
3,106,72,0,0,25.8,0.207,27,0
4,97,60,23,0,28.2,0.443,22,0
3,100,68,23,81,31.6,0.949,28,0
6,162,62,0,0,24.3,0.178,50,1
4,111,72,47,207,37.1,1.390,56,1
5,101,48,27,140,28.7,0.369,36,1
4,83,86,19,0,29.3,0.317,34,0
2,115,70,30,96,34.6,0.529,32,1
2,90,60,0,0,23.5,0.191,25,0
6,93,50,30,64,28.7,0.356,23,0
8,143,78,47,223,37.7,0.378,36,1
2,100,66,20,90,32.9,0.867,28,1
7,150,66,42,342,34.7,0.718,42,0
4,103,60,33,192,24.0,0.966,33,0
7,168,88,42,321,38.2,0.787,40,1
0,135,68,42,250,42.3,0.365,24,1
2,74,0,0,0,0.0,0.102,22,0
6,125,68,30,120,30.0,0.464,32,0
10,115,0,0,0,35.3,0.134,29,0
7,100,0,0,0,30.0,0.484,32,1
5,130,82,0,0,39.1,0.956,37,1
7,159,64,0,0,27.4,0.294,40,0
9,125,70,30,0,36.4,0.738,48,1
1,73,50,10,0,23.0,0.248,21,0
3,75,68,22,0,31.6,0.537,26,0
3,158,76,36,245,31.6,0.851,28,1
1,97,66,15,140,23.2,0.487,22,0
0,193,80,25,10,37.6,0.936,22,0
2,108,62,32,56,25.2,0.128,21,0
2,73,64,18,88,18.2,0.503,25,0
8,131,52,0,0,27.4,0.221,52,0
5,139,60,24,0,26.4,0.223,25,0
2,87,88,0,0,32.7,0.217,21,0
3,89,74,16,85,30.4,0.551,38,0
0,105,90,0,0,29.6,0.197,46,0
4,110,76,20,100,28.4,0.118,27,0
8,111,70,0,0,30.1,0.283,33,1
0,144,0,0,0,0.0,0.804,30,0
2,111,60,0,0,26.2,0.343,23,0
0,120,69,32,0,36.4,0.736,24,1
6,111,64,39,0,34.2,0.260,24,0
1,72,72,25,0,31.2,0.295,44,0
7,168,88,42,321,38.2,0.787,40,1
8,130,82,42,0,38.4,0.503,45,1
1,77,56,30,56,33.3,1.251,24,0
2,87,58,16,52,32.7,0.166,25,0
2,152,60,0,0,24.6,0.226,50,0
4,109,64,44,99,34.6,0.415,34,0
6,125,68,30,120,30.0,0.464,32,0
3,139,54,0,0,25.6,0.402,22,1
1,103,80,11,82,19.4,0.491,22,0
3,152,72,30,97,27.6,0.761,35,0
9,119,80,35,0,29.0,0.263,29,1
0,154,70,29,218,32.1,0.613,24,0
0,154,70,29,218,32.1,0.613,24,0
6,107,88,0,0,36.8,0.727,31,0
5,132,80,0,0,26.8,0.186,69,0
3,83,58,31,18,34.3,0.336,25,0
6,147,80,0,0,29.5,0.178,50,1
7,178,84,0,0,39.9,0.331,41,1
3,85,82,31,0,30.2,0.630,27,0
2,116,74,0,0,25.6,0.201,30,0
0,119,64,18,92,34.9,0.725,23,0
2,71,70,27,0,28.0,0.586,22,0
2,158,90,0,0,31.6,0.805,66,1
1,77,56,30,56,33.3,1.251,24,0
3,158,76,36,245,31.6,0.851,28,1
1,73,50,10,0,23.0,0.248,21,0
3,71,88,0,0,30.4,0.247,22,0
1,97,68,21,0,27.2,1.095,22,0
0,148,58,11,179,39.9,0.160,45,0
4,100,70,37,100,40.2,0.205,25,0
9,94,80,45,0,45.6,0.360,34,0
4,103,60,33,192,24.0,0.966,33,0
7,196,90,0,0,39.8,0.451,41,1
2,116,74,0,0,25.6,0.201,30,0
4,61,96,0,0,35.5,0.925,52,0
1,80,55,0,0,19.1,0.258,21,0
8,134,72,37,0,26.0,0.635,44,1
3,66,72,0,0,24.2,0.261,25,0
5,117,92,0,0,34.1,0.337,38,0
1,93,70,31,0,30.4,0.315,23,0
2,104,68,0,0,35.8,0.218,37,0
6,123,74,0,0,32.1,0.286,43,1
4,103,60,33,192,24.0,0.966,33,0
9,148,84,0,0,30.0,0.316,47,0
3,91,78,36,0,39.6,0.498,28,0
9,134,74,33,60,25.9,0.460,81,0
0,98,0,0,0,0.0,0.134,22,0
3,76,68,15,112,25.0,0.628,39,0
4,99,68,38,0,32.8,0.145,33,0
11,143,94,33,146,36.6,0.254,51,1
5,99,54,28,83,34.0,0.499,30,0
4,109,64,44,99,34.6,0.415,34,0
2,108,62,32,56,25.2,0.128,21,0
1,97,60,23,0,28.2,0.443,22,0
3,127,88,35,0,44.2,0.396,34,0
5,130,82,0,0,39.1,0.956,37,1
5,116,74,0,0,25.6,0.201,30,0
3,99,54,19,86,25.6,0.154,24,0
5,129,62,36,46,32.2,0.515,41,0
0,117,80,31,53,45.2,0.089,24,0
8,186,90,35,225,34.5,0.423,37,1
0,105,84,0,0,27.9,0.741,62,1
3,78,50,32,88,31.0,0.248,26,1
2,107,74,30,100,33.6,0.404,23,0
5,117,86,30,105,39.1,0.251,42,0
2,91,62,27,0,29.7,0.583,25,0
7,168,88,42,321,38.2,0.787,40,1
2,74,0,0,0,0.0,0.102,22,0
2,197,70,45,543,30.5,0.158,53,1
0,113,80,16,0,31.0,0.874,21,0
1,133,102,28,140,32.8,0.234,45,1
3,91,78,36,0,39.6,0.498,28,0
1,98,60,25,110,25.0,1.000,28,0
3,139,54,0,0,25.6,0.402,22,1
9,164,84,21,0,30.8,0.831,32,1
5,175,96,0,0,36.5,0.746,48,1
7,119,60,0,0,32.0,0.294,49,0
4,146,78,0,0,38.5,0.520,67,1
4,105,72,0,0,22.9,0.143,35,0
2,84,72,38,120,28.6,0.906,26,1
0,155,56,28,0,28.7,0.481,37,0
7,196,90,0,0,39.8,0.451,41,1
3,119,66,28,0,31.9,0.728,37,0
1,97,66,15,140,23.2,0.487,22,0
8,100,76,0,0,38.7,0.190,42,0
5,116,74,0,0,25.6,0.201,30,0
4,103,60,33,192,24.0,0.966,33,0
4,99,68,38,0,32.8,0.145,33,0
6,170,80,46,0,29.0,0.263,52,0
3,112,62,0,0,26.4,0.177,50,0
7,103,68,40,0,46.2,0.665,33,0
4,143,82,26,270,32.4,0.834,23,1
3,75,68,22,0,31.6,0.537,26,0
4,102,60,22,0,26.0,0.440,21,0
8,101,76,48,58,35.8,0.467,52,1
1,95,66,13,38,19.6,0.334,25,0
1,99,62,31,0,36.1,0.492,31,0
4,105,72,0,0,22.9,0.143,35,0
3,106,72,0,0,25.8,0.207,27,0
5,99,54,28,83,34.0,0.499,30,0
2,168,64,0,0,36.4,0.554,38,1
9,127,70,39,0,33.7,0.537,54,0
5,130,82,0,0,39.1,0.956,37,1
1,111,94,0,0,32.8,0.265,45,0
6,93,50,30,64,28.7,0.356,23,0
4,143,82,26,270,32.4,0.834,23,1
1,127,84,0,0,22.0,0.620,32,0
9,112,82,24,0,28.2,1.282,50,1
2,122,76,27,200,35.9,0.207,26,0
2,95,54,14,88,26.1,0.748,22,0
0,135,68,42,250,42.3,0.365,24,1
4,129,60,12,231,27.5,0.527,31,0
6,119,50,22,176,27.1,1.318,33,1
3,91,62,0,0,25.3,0.233,21,0
1,91,54,25,100,25.2,0.234,23,0
6,148,72,35,0,33.6,0.627,50,1
1,85,66,29,0,26.6,0.351,31,0
8,183,64,0,0,23.3,0.672,32,1
1,89,66,23,94,28.1,0.167,21,0
0,137,40,35,168,43.1,2.288,33,1
5,116,74,0,0,25.6,0.201,30,0
3,78,50,32,88,31.0,0.248,26,1
2,197,70,45,543,30.5,0.158,53,1
8,125,96,0,0,0.0,0.232,54,1
4,110,92,0,0,37.6,0.191,30,0
10,168,74,0,0,38.0,0.537,34,1
10,139,80,0,0,27.1,1.441,57,0
1,189,60,23,846,30.1,0.398,59,1
5,166,72,19,175,25.8,0.587,51,1
7,100,0,0,0,30.0,0.484,32,1
0,118,84,47,230,45.8,0.551,31,1
7,107,74,0,0,29.6,0.254,31,1
1,103,30,38,83,43.3,0.183,33,0
1,115,70,30,96,34.6,0.529,32,1
3,126,88,41,235,39.3,0.704,27,0
8,99,84,0,0,35.4,0.388,50,0
7,196,90,0,0,39.8,0.451,41,1
9,119,80,35,0,29.0,0.263,29,1
11,143,94,33,146,36.6,0.254,51,1
10,125,70,26,115,31.1,0.205,41,1
7,147,76,0,0,39.4,0.257,43,1
1,97,66,15,140,23.2,0.487,22,0
13,145,82,19,110,22.2,0.245,57,0
5,117,92,0,0,34.1,0.337,38,0
3,158,76,36,245,31.6,0.851,28,1
3,88,58,11,54,24.8,0.267,22,0
6,92,92,0,0,19.9,0.188,28,0
10,122,78,31,0,27.6,0.512,45,0
4,103,60,33,192,24.0,0.966,33,0
11,138,76,0,0,33.2,0.420,35,0
9,102,76,37,0,32.9,0.665,46,1
2,90,68,42,0,38.2,0.503,27,1
4,111,72,47,207,37.1,1.390,56,1
3,180,64,25,70,34.0,0.271,26,0
7,133,84,0,0,40.2,0.696,37,0
7,106,92,18,0,22.7,0.235,48,0
9,171,110,24,240,45.4,0.721,54,1
7,159,64,0,0,27.4,0.294,40,0
0,180,66,39,0,42.0,1.893,25,1
1,146,56,0,0,29.7,0.564,29,0
2,71,70,27,0,28.0,0.586,22,0
7,103,66,32,0,39.1,0.344,31,1
7,105,0,0,0,0.0,0.305,24,0
1,103,80,11,82,19.4,0.491,22,0
1,101,50,15,36,24.2,0.526,26,0
5,88,66,21,23,24.4,0.342,30,0
8,176,90,34,300,33.7,0.467,58,1
7,150,66,42,342,34.7,0.718,42,0
1,73,50,10,0,23.0,0.248,21,0
7,187,68,39,304,37.7,0.254,41,1
0,100,88,60,110,46.8,0.962,31,0
0,146,82,0,0,40.5,1.781,44,0
0,105,64,41,142,41.5,0.173,22,0
2,84,0,0,0,0.0,0.304,21,0
8,133,72,0,0,32.9,0.270,39,1
5,44,62,0,0,25.0,0.587,36,0
2,141,58,34,128,25.4,0.699,24,0
7,114,66,0,0,32.8,0.258,42,1
5,99,54,28,83,34.0,0.499,30,0
0,109,88,30,0,32.5,0.855,38,1
2,109,92,0,0,42.7,0.845,54,0
1,95,66,13,38,19.6,0.334,25,0
4,146,85,27,100,28.9,0.189,27,0
8,112,72,0,0,23.6,0.840,58,0
0,138,0,0,0,36.3,0.933,25,1
3,102,44,20,94,30.8,0.400,26,0
3,87,60,18,0,21.8,0.444,21,0
4,156,75,0,0,48.3,0.238,32,1
6,93,50,30,64,28.7,0.356,23,0
4,61,96,0,0,35.5,0.925,52,0
4,129,60,12,231,27.5,0.527,31,0
3,138,62,44,208,24.0,0.290,22,0
0,98,64,17,0,20.6,0.507,24,0
9,154,78,30,100,30.9,0.164,45,0
7,103,68,40,0,46.2,0.665,33,0
1,126,60,0,0,30.1,0.349,47,1
8,99,76,0,0,35.4,0.388,50,0
4,112,76,0,0,38.3,0.234,35,0
8,167,106,46,231,37.6,0.165,43,1
1,78,76,32,0,36.9,0.264,29,0
3,90,78,0,0,42.7,0.559,21,0
9,164,84,21,0,30.8,0.831,32,1
5,145,80,0,0,37.9,0.637,41,0
1,77,56,30,56,33.3,1.251,24,0
4,110,76,20,100,28.4,0.118,27,0
2,74,0,0,0,0.0,0.102,22,0
6,149,88,29,260,28.5,0.687,55,0
9,134,74,33,60,25.9,0.460,81,0
1,73,50,10,0,23.0,0.248,21,0
6,107,88,0,0,36.8,0.727,31,0
1,65,76,0,0,24.8,0.248,23,0
8,130,82,42,0,38.4,0.503,45,1
8,187,90,34,0,37.7,0.956,55,1
5,100,76,27,0,39.4,0.560,35,0
4,68,120,0,0,35.1,0.258,22,0
9,112,82,24,0,28.2,1.282,50,1
0,119,0,0,0,32.4,0.141,24,1
2,112,68,22,94,34.1,0.315,26,0
2,116,74,0,0,25.6,0.201,30,0
2,70,56,27,0,31.7,0.503,26,0
3,91,78,36,0,39.6,0.498,28,0
4,105,72,0,0,22.9,0.143,35,0
4,189,110,31,0,28.5,0.680,37,0
0,147,85,54,0,42.8,0.375,22,0
4,112,72,0,0,23.1,0.289,22,0
7,142,60,33,0,28.8,0.687,61,0
4,109,64,44,99,34.6,0.415,34,0
1,123,88,39,182,40.9,0.969,35,0
7,136,90,0,0,29.9,0.210,50,0
3,106,72,0,0,25.8,0.207,27,0
3,103,72,30,152,27.6,0.730,27,0
1,128,88,39,110,36.5,1.057,37,1
3,115,66,39,140,38.1,0.150,28,0
5,126,130,0,0,30.1,0.473,47,0
2,106,64,35,119,30.5,1.400,34,0
2,102,86,36,120,45.5,0.127,23,1
6,125,68,30,120,30.0,0.464,32,0
0,126,86,27,120,27.4,0.515,21,0
2,168,64,0,0,36.4,0.554,38,1
3,152,72,30,97,27.6,0.761,35,0
6,156,60,0,0,26.5,0.483,60,0
0,95,80,45,92,36.5,0.330,26,0
3,71,88,0,0,30.4,0.247,22,0
4,99,68,38,0,32.8,0.145,33,0
4,107,68,0,0,29.4,0.251,28,0
8,100,76,0,0,38.7,0.190,42,0
1,97,105,0,0,30.1,0.218,21,0
0,123,72,0,0,36.3,0.258,52,1
6,115,60,0,0,34.3,0.169,48,0
4,99,96,0,0,24.6,0.847,35,0
1,119,88,41,170,45.3,0.507,26,0
4,114,64,0,0,28.9,0.126,24,0
3,182,74,0,0,30.5,0.345,46,1
7,68,49,26,135,42.1,0.398,52,1
0,162,76,56,100,53.2,0.759,25,1
1,72,72,25,0,31.2,0.295,44,0
8,101,76,48,58,35.8,0.467,52,1
4,97,60,23,0,28.2,0.443,22,0
3,83,58,31,18,34.3,0.336,25,0
2,109,75,25,0,31.9,0.253,27,0
6,119,50,22,176,27.1,1.318,33,1
0,99,56,0,0,28.2,0.201,25,0
2,65,60,35,0,24.9,0.243,21,0
5,130,82,0,0,39.1,0.956,37,1
2,87,58,16,52,32.7,0.166,25,0
5,175,96,0,0,36.5,0.746,48,1
4,108,80,0,0,26.5,0.144,37,0
9,125,70,30,0,36.4,0.738,48,1
1,157,72,21,168,25.6,0.123,24,0
12,140,85,33,0,37.4,0.244,41,0
5,147,75,0,0,29.9,0.434,28,0
1,97,66,15,140,23.2,0.487,22,0
6,107,88,0,0,36.8,0.727,31,0
0,189,104,25,0,34.3,0.435,41,1
4,108,80,0,0,26.5,0.144,37,0
1,121,78,39,74,39.0,0.261,28,0
3,108,62,24,0,26.0,0.223,25,0
0,181,88,44,510,43.3,0.222,26,1
8,154,78,32,0,32.4,0.443,45,1
1,128,88,39,110,36.5,1.057,37,1
7,137,90,41,0,32.0,0.391,39,0
0,123,72,0,0,36.3,0.258,52,1
1,106,76,0,0,37.5,0.197,26,0
6,190,92,0,0,35.5,0.278,66,1
2,88,58,26,16,28.4,0.766,22,0
9,170,74,31,0,44.0,0.403,43,1
9,89,62,0,0,22.5,0.142,33,0
13,145,82,19,110,22.2,0.245,57,0
2,197,70,45,543,30.5,0.158,53,1
4,99,68,38,0,32.8,0.145,33,0
3,158,76,36,245,31.6,0.851,28,1
6,92,92,0,0,19.9,0.188,28,0
10,122,78,31,0,27.6,0.512,45,0
4,103,60,33,192,24.0,0.966,33,0
0,100,88,60,110,46.8,0.962,31,0
9,171,110,24,240,45.4,0.721,54,1
8,126,88,36,108,38.5,0.349,49,0
5,123,74,40,77,34.1,0.269,28,0
0,107,60,25,0,26.4,0.133,23,0
6,111,64,39,0,34.2,0.260,24,0
3,76,68,15,112,25.0,0.628,39,0
8,74,64,46,0,21.0,0.672,29,1
5,116,74,0,0,25.6,0.201,30,0
1,126,60,0,0,30.1,0.349,47,1
3,78,50,32,88,31.0,0.248,26,1
1,101,62,0,0,21.9,0.336,28,0
7,168,88,42,321,38.2,0.787,40,1
7,159,64,0,0,27.4,0.294,40,0
0,155,84,0,0,34.4,0.304,23,0
0,118,84,47,230,45.8,0.551,31,1
3,156,84,35,220,28.3,0.884,27,1
7,103,66,32,0,39.1,0.344,31,1
6,111,74,0,0,26.4,0.206,40,0
6,139,80,57,0,52.7,1.695,22,1
4,112,72,0,0,23.1,0.289,22,0
5,100,76,27,0,39.4,0.560,35,0
9,127,70,39,0,33.7,0.537,54,0
4,129,86,20,270,35.1,0.231,23,0
6,170,80,46,0,29.0,0.263,52,0
1,97,60,23,0,28.2,0.443,22,0
7,181,84,21,192,35.9,0.586,51,1
0,117,80,31,53,45.2,0.089,24,0
3,83,58,31,18,34.3,0.336,25,0
2,141,58,34,128,25.4,0.699,24,0
1,97,68,21,0,27.2,1.095,22,0
4,83,86,19,0,29.3,0.317,34,0
0,101,64,17,0,21.0,0.252,21,0
3,112,78,50,140,39.4,0.175,24,0
4,152,76,0,0,33.2,0.731,47,1
2,109,75,25,0,31.9,0.253,27,0
10,139,80,0,0,27.1,1.441,57,0
3,106,72,0,0,25.8,0.207,27,0
4,97,60,23,0,28.2,0.443,22,0
3,100,68,23,81,31.6,0.949,28,0
6,162,62,0,0,24.3,0.178,50,1
4,111,72,47,207,37.1,1.390,56,1
5,101,48,27,140,28.7,0.369,36,1
4,83,86,19,0,29.3,0.317,34,0
2,115,70,30,96,34.6,0.529,32,1
2,90,60,0,0,23.5,0.191,25,0
6,93,50,30,64,28.7,0.356,23,0
8,143,78,47,223,37.7,0.378,36,1
2,100,66,20,90,32.9,0.867,28,1
7,150,66,42,342,34.7,0.718,42,0
4,103,60,33,192,24.0,0.966,33,0
7,168,88,42,321,38.2,0.787,40,1
0,135,68,42,250,42.3,0.365,24,1
2,74,0,0,0,0.0,0.102,22,0
6,125,68,30,120,30.0,0.464,32,0
10,115,0,0,0,35.3,0.134,29,0
7,100,0,0,0,30.0,0.484,32,1
5,130,82,0,0,39.1,0.956,37,1
7,159,64,0,0,27.4,0.294,40,0
9,125,70,30,0,36.4,0.738,48,1
1,73,50,10,0,23.0,0.248,21,0
3,75,68,22,0,31.6,0.537,26,0
3,158,76,36,245,31.6,0.851,28,1
0,193,80,25,10,37.6,0.936,22,0
2,108,62,32,56,25.2,0.128,21,0
2,73,64,18,88,18.2,0.503,25,0
8,131,52,0,0,27.4,0.221,52,0
5,139,60,24,0,26.4,0.223,25,0
2,87,88,0,0,32.7,0.217,21,0
3,89,74,16,85,30.4,0.551,38,0
0,105,90,0,0,29.6,0.197,46,0
4,110,76,20,100,28.4,0.118,27,0
8,111,70,0,0,30.1,0.283,33,1
0,144,0,0,0,0.0,0.804,30,0
2,111,60,0,0,26.2,0.343,23,0
0,120,69,32,0,36.4,0.736,24,1
6,111,64,39,0,34.2,0.260,24,0
1,72,72,25,0,31.2,0.295,44,0
7,168,88,42,321,38.2,0.787,40,1
8,130,82,42,0,38.4,0.503,45,1
1,77,56,30,56,33.3,1.251,24,0
2,87,58,16,52,32.7,0.166,25,0
2,152,60,0,0,24.6,0.226,50,0
4,109,64,44,99,34.6,0.415,34,0
6,125,68,30,120,30.0,0.464,32,0
3,139,54,0,0,25.6,0.402,22,1
1,103,80,11,82,19.4,0.491,22,0
3,152,72,30,97,27.6,0.761,35,0
9,119,80,35,0,29.0,0.263,29,1
0,154,70,29,218,32.1,0.613,24,0
6,107,88,0,0,36.8,0.727,31,0
5,132,80,0,0,26.8,0.186,69,0
3,83,58,31,18,34.3,0.336,25,0
6,147,80,0,0,29.5,0.178,50,1
7,178,84,0,0,39.9,0.331,41,1
3,85,82,31,0,30.2,0.630,27,0
2,116,74,0,0,25.6,0.201,30,0
0,119,64,18,92,34.9,0.725,23,0
2,71,70,27,0,28.0,0.586,22,0
2,158,90,0,0,31.6,0.805,66,1
3,158,76,36,245,31.6,0.851,28,1
1,73,50,10,0,23.0,0.248,21,0
3,71,88,0,0,30.4,0.247,22,0
1,97,68,21,0,27.2,1.095,22,0
0,148,58,11,179,39.9,0.160,45,0
4,100,70,37,100,40.2,0.205,25,0
9,94,80,45,0,45.6,0.360,34,0
4,103,60,33,192,24.0,0.966,33,0
7,196,90,0,0,39.8,0.451,41,1
2,116,74,0,0,25.6,0.201,30,0
4,61,96,0,0,35.5,0.925,52,0
1,80,55,0,0,19.1,0.258,21,0
8,134,72,37,0,26.0,0.635,44,1
3,66,72,0,0,24.2,0.261,25,0
5,117,92,0,0,34.1,0.337,38,0
1,93,70,31,0,30.4,0.315,23,0
2,104,68,0,0,35.8,0.218,37,0
6,123,74,0,0,32.1,0.286,43,1
9,148,84,0,0,30.0,0.316,47,0
3,91,78,36,0,39.6,0.498,28,0
9,134,74,33,60,25.9,0.460,81,0
0,98,0,0,0,0.0,0.134,22,0
3,76,68,15,112,25.0,0.628,39,0
4,99,68,38,0,32.8,0.145,33,0
11,143,94,33,146,36.6,0.254,51,1
5,99,54,28,83,34.0,0.499,30,0
4,109,64,44,99,34.6,0.415,34,0
2,108,62,32,56,25.2,0.128,21,0
1,97,60,23,0,28.2,0.443,22,0
3,127,88,35,0,44.2,0.396,34,0
5,130,82,0,0,39.1,0.956,37,1
5,116,74,0,0,25.6,0.201,30,0
3,99,54,19,86,25.6,0.154,24,0
5,129,62,36,46,32.2,0.515,41,0
0,117,80,31,53,45.2,0.089,24,0
8,186,90,35,225,34.5,0.423,37,1
0,105,84,0,0,27.9,0.741,62,1
3,78,50,32,88,31.0,0.248,26,1
2,107,74,30,100,33.6,0.404,23,0
5,117,86,30,105,39.1,0.251,42,0
2,91,62,27,0,29.7,0.583,25,0
2,74,0,0,0,0.0,0.102,22,0
2,197,70,45,543,30.5,0.158,53,1
0,113,80,16,0,31.0,0.874,21,0
1,133,102,28,140,32.8,0.234,45,1
3,91,78,36,0,39.6,0.498,28,0
1,98,60,25,110,25.0,1.000,28,0
3,139,54,0,0,25.6,0.402,22,1
9,164,84,21,0,30.8,0.831,32,1
5,175,96,0,0,36.5,0.746,48,1
7,119,60,0,0,32.0,0.294,49,0
4,146,78,0,0,38.5,0.520,67,1
4,105,72,0,0,22.9,0.143,35,0
2,84,72,38,120,28.6,0.906,26,1
0,155,56,28,0,28.7,0.481,37,0
3,119,66,28,0,31.9,0.728,37,0
5,116,74,0,0,25.6,0.201,30,0
3,78,50,32,88,31.0,0.248,26,1
1,101,62,0,0,21.9,0.336,28,0
7,168,88,42,321,38.2,0.787,40,1
0,155,84,0,0,34.4,0.304,23,0
0,118,84,47,230,45.8,0.551,31,1
3,156,84,35,220,28.3,0.884,27,1
7,103,66,32,0,39.1,0.344,31,1
4,143,82,26,270,32.4,0.834,23,1
3,75,68,22,0,31.6,0.537,26,0
4,102,60,22,0,26.0,0.440,21,0
8,101,76,48,58,35.8,0.467,52,1
1,95,66,13,38,19.6,0.334,25,0
1,99,62,31,0,36.1,0.492,31,0"""

def load_data():
    """Load Pima Indians Diabetes dataset from embedded CSV."""
    df = pd.read_csv(StringIO(DIABETES_CSV))
    print(f"[OK] Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"  Diabetic: {df['Outcome'].sum()} | Non-diabetic: {(df['Outcome']==0).sum()}")
    return df


def clean_data(df):
    """Replace biologically impossible zero values with column medians."""
    cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    df = df.copy()
    for col in cols_with_zeros:
        median = df[col].replace(0, pd.NA).median()
        zeros_count = (df[col] == 0).sum()
        df[col] = df[col].replace(0, median)
        print(f"  {col}: replaced {zeros_count} zeros with median={median:.1f}")
    print(f"[OK] Data cleaning complete")
    return df


def train_models(X_train, X_test, y_train, y_test):
    """Train 5 classifiers and evaluate on test set."""
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Decision Tree":       DecisionTreeClassifier(random_state=42),
        "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
        "KNN":                 KNeighborsClassifier(n_neighbors=5),
        "SVM":                 SVC(probability=True, random_state=42),
    }

    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        results[name] = {
            "model":     model,
            "accuracy":  round(acc * 100, 1),
            "precision": round(prec * 100, 1),
            "recall":    round(rec * 100, 1),
            "f1_score":  round(f1 * 100, 1),
        }
        print(f"  {name:<25} acc={acc*100:.1f}%  prec={prec*100:.1f}%  rec={rec*100:.1f}%  f1={f1*100:.1f}%")
    return results


def save_artifacts(best_model, scaler, metrics, model_comparison):
    """Persist model, scaler, and evaluation metrics as JSON."""
    joblib.dump(best_model, "diabetes_model.pkl")
    joblib.dump(scaler,     "scaler.pkl")

    stats = {
        "accuracy":        metrics["accuracy"],
        "precision":       metrics["precision"],
        "recall":          metrics["recall"],
        "f1_score":        metrics["f1_score"],
        "models_compared": {k: v["accuracy"] for k, v in model_comparison.items()},
    }
    with open("model_stats.json", "w") as f:
        json.dump(stats, f, indent=2)

    print("[OK] Saved: diabetes_model.pkl  scaler.pkl  model_stats.json")


def main():
    print("\n" + "="*50)
    print("  DiabetesAI - ML Training Pipeline")
    print("="*50 + "\n")

    # ── STEP 1: Load data ──────────────────────────────
    df = load_data()

    # ── STEP 2: Clean data ─────────────────────────────
    print("\n[STEP 2] Data Cleaning")
    df = clean_data(df)

    # ── STEP 3 & 4: Feature split + scaling ───────────
    print("\n[STEP 3-4] Features & Scaling")
    X = df.drop("Outcome", axis=1)
    y = df["Outcome"]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    print(f"[OK] StandardScaler applied to {X.shape[1]} features")

    # ── STEP 5: Train/test split ───────────────────────
    print("\n[STEP 5] Train/Test Split (80/20)")
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )
    print(f"[OK] Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

    # ── STEP 6: Train all models ───────────────────────
    print("\n[STEP 6] Model Training & Evaluation")
    results = train_models(X_train, X_test, y_train, y_test)

    # ── STEP 7: Select best model ──────────────────────
    print("\n[STEP 7] Best Model Selection")
    # We prefer Random Forest because it gives smooth probabilities for risk scoring
    if "Random Forest" in results:
        best_name = "Random Forest"
    else:
        best_name = max(results, key=lambda k: results[k]["accuracy"])
    best_info = results[best_name]
    print(f"[OK] Winner: {best_name} ({best_info['accuracy']}% accuracy)")

    # Detailed report on best model
    print("\n[STEP 8] Full Evaluation Report")
    y_pred = best_info["model"].predict(X_test)
    print(classification_report(y_test, y_pred, target_names=["Non-Diabetic", "Diabetic"]))
    cm = confusion_matrix(y_test, y_pred)
    print(f"Confusion Matrix:\n{cm}")

    # ── STEP 8: Save artifacts ─────────────────────────
    print("\n[STEP 9] Saving Artifacts")
    save_artifacts(best_info["model"], scaler, best_info, results)

    print("\n" + "="*50)
    print("  Training Complete!")
    print("="*50 + "\n")


if __name__ == "__main__":
    main()


## 3. Additional Metrics to Display

Add these after model prediction:

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC Score
- Specificity
- Sensitivity
- Confusion Matrix
- Classification Report
- Cross Validation Score
- Feature Importance (Random Forest)


In [ ]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix\n", cm)

tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)

print("Specificity:", specificity)
print("Sensitivity:", sensitivity)

print("\nClassification Report\n")
print(classification_report(y_test, y_pred))


## 4. Feature Importance

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

if hasattr(best_model, 'feature_importances_'):
    imp = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)

    display(imp)

    plt.figure(figsize=(8,5))
    plt.barh(imp['Feature'], imp['Importance'])
    plt.title('Feature Importance')
    plt.tight_layout()
    plt.show()
